In [ ]:
import tensorflow as tf
import numpy as np
import matplotlib.pyplot as plt

tf.keras.backend.set_floatx("float32")
np.random.seed(42)
tf.random.set_seed(42)

# =========================
# Physical parameters
# =========================
hbar = 1.0
m = 1.0
k0 = 2.0

# Domain
x_min, x_max = -5.0, 5.0
t_min, t_max = 0.0, 2.0

# Evaluation grid
N_x, N_t = 200, 200
x_eval = np.linspace(x_min, x_max, N_x, dtype=np.float32)
t_eval = np.linspace(t_min, t_max, N_t, dtype=np.float32)
X_eval, T_eval = np.meshgrid(x_eval, t_eval)
XT_eval = np.hstack([X_eval.reshape(-1, 1), T_eval.reshape(-1, 1)]).astype(np.float32)
XT_eval_tf = tf.convert_to_tensor(XT_eval)

# =========================
# Initial condition
# ψ(x,0) = exp(-x^2) exp(i k0 x)
# =========================
def psi0_np(x):
    return np.exp(-x**2) * np.exp(1j * k0 * x)

psi0 = psi0_np(x_eval)
psi0_real = psi0.real.astype(np.float32)
psi0_imag = psi0.imag.astype(np.float32)

# =========================
# PINN model
# =========================
def build_model():
    inputs = tf.keras.Input(shape=(2,))
    x = tf.keras.layers.Dense(64, activation="tanh")(inputs)
    x = tf.keras.layers.Dense(64, activation="tanh")(x)
    x = tf.keras.layers.Dense(64, activation="tanh")(x)
    x = tf.keras.layers.Dense(64, activation="tanh")(x)
    outputs = tf.keras.layers.Dense(2)(x)   # [Re(ψ), Im(ψ)]
    return tf.keras.Model(inputs, outputs)

model = build_model()

# =========================
# Sampling utilities
# =========================
def sample_interior(n):
    x = np.random.uniform(x_min, x_max, (n, 1)).astype(np.float32)
    t = np.random.uniform(t_min, t_max, (n, 1)).astype(np.float32)
    return tf.convert_to_tensor(np.hstack([x, t]))

def sample_initial(n):
    x = np.random.uniform(x_min, x_max, (n, 1)).astype(np.float32)
    t = np.zeros((n, 1), dtype=np.float32)
    xt = np.hstack([x, t])
    psi = psi0_np(x[:, 0])
    y = np.column_stack([psi.real, psi.imag]).astype(np.float32)
    return tf.convert_to_tensor(xt), tf.convert_to_tensor(y)

def sample_boundary(n):
    t = np.random.uniform(t_min, t_max, (n, 1)).astype(np.float32)

    x_left = np.full((n, 1), x_min, dtype=np.float32)
    x_right = np.full((n, 1), x_max, dtype=np.float32)

    xt_left = np.hstack([x_left, t])
    xt_right = np.hstack([x_right, t])

    return tf.convert_to_tensor(xt_left), tf.convert_to_tensor(xt_right)

# =========================
# PDE residual
# i ψ_t + (ħ²/2m) ψ_xx = 0
#
# ψ = u + i v
# => v_t + (ħ²/2m) u_xx = 0
# => -u_t + (ħ²/2m) v_xx = 0
# =========================
@tf.function
def pde_residual(xt):
    with tf.GradientTape(persistent=True) as tape2:
        tape2.watch(xt)
        with tf.GradientTape(persistent=True) as tape1:
            tape1.watch(xt)
            psi = model(xt)
            u = psi[:, 0:1]
            v = psi[:, 1:2]

        du = tape1.gradient(u, xt)
        dv = tape1.gradient(v, xt)

        u_x = du[:, 0:1]
        u_t = du[:, 1:2]
        v_x = dv[:, 0:1]
        v_t = dv[:, 1:2]

    d2u = tape2.gradient(u_x, xt)
    d2v = tape2.gradient(v_x, xt)

    u_xx = d2u[:, 0:1]
    v_xx = d2v[:, 0:1]

    del tape1
    del tape2

    coeff = hbar**2 / (2.0 * m)

    r1 = v_t + coeff * u_xx
    r2 = -u_t + coeff * v_xx

    return r1, r2

# =========================
# Loss function
# =========================
lambda_ic = 10.0
lambda_bc = 1.0

@tf.function
def compute_loss(xt_interior, xt_ic, y_ic, xt_left, xt_right):
    r1, r2 = pde_residual(xt_interior)
    loss_pde = tf.reduce_mean(tf.square(r1)) + tf.reduce_mean(tf.square(r2))

    psi_ic_pred = model(xt_ic)
    loss_ic = tf.reduce_mean(tf.square(psi_ic_pred - y_ic))

    psi_left = model(xt_left)
    psi_right = model(xt_right)
    loss_bc = tf.reduce_mean(tf.square(psi_left)) + tf.reduce_mean(tf.square(psi_right))

    total_loss = loss_pde + lambda_ic * loss_ic + lambda_bc * loss_bc
    return total_loss, loss_pde, loss_ic, loss_bc

# =========================
# Training
# =========================
optimizer = tf.keras.optimizers.Adam(learning_rate=1e-3)

loss_history = []
pde_history = []
ic_history = []
bc_history = []

@tf.function
def train_step(xt_interior, xt_ic, y_ic, xt_left, xt_right):
    with tf.GradientTape() as tape:
        total_loss, loss_pde, loss_ic, loss_bc = compute_loss(
            xt_interior, xt_ic, y_ic, xt_left, xt_right
        )
    grads = tape.gradient(total_loss, model.trainable_variables)
    optimizer.apply_gradients(zip(grads, model.trainable_variables))
    return total_loss, loss_pde, loss_ic, loss_bc

epochs = 5000
N_f = 4000   # interior collocation points
N_ic = 400   # initial-condition points
N_bc = 400   # boundary points

for epoch in range(epochs):
    xt_f = sample_interior(N_f)
    xt_ic, y_ic = sample_initial(N_ic)
    xt_left, xt_right = sample_boundary(N_bc)

    total_loss, loss_pde, loss_ic, loss_bc = train_step(
        xt_f, xt_ic, y_ic, xt_left, xt_right
    )

    loss_history.append(float(total_loss.numpy()))
    pde_history.append(float(loss_pde.numpy()))
    ic_history.append(float(loss_ic.numpy()))
    bc_history.append(float(loss_bc.numpy()))

    if epoch % 500 == 0:
        print(
            f"Epoch {epoch:5d} | "
            f"Total = {total_loss.numpy():.3e} | "
            f"PDE = {loss_pde.numpy():.3e} | "
            f"IC = {loss_ic.numpy():.3e} | "
            f"BC = {loss_bc.numpy():.3e}"
        )

# =========================
# Prediction on grid
# =========================
psi_pred = model(XT_eval_tf).numpy()
u_pred = psi_pred[:, 0].reshape(N_t, N_x)
v_pred = psi_pred[:, 1].reshape(N_t, N_x)

psi_modulus = np.sqrt(u_pred**2 + v_pred**2)
psi_density = u_pred**2 + v_pred**2   # |ψ|²

# =========================
# Diagnostics: normalization and center of mass
# =========================
dx = x_eval[1] - x_eval[0]

norm_t = np.trapz(psi_density, x_eval, axis=1)
x_mean_t = np.trapz(X_eval * psi_density, x_eval, axis=1) / np.maximum(norm_t, 1e-8)

# =========================
# Plots
# =========================
plt.figure(figsize=(10, 6))
plt.imshow(
    psi_modulus,
    extent=[x_min, x_max, t_min, t_max],
    origin="lower",
    aspect="auto",
    cmap="inferno"
)
plt.colorbar(label=r"$|\psi|$")
plt.xlabel("x")
plt.ylabel("t")
plt.title(r"PINN prediction for $|\psi(x,t)|$")
plt.show()

plt.figure(figsize=(10, 6))
plt.imshow(
    psi_density,
    extent=[x_min, x_max, t_min, t_max],
    origin="lower",
    aspect="auto",
    cmap="inferno"
)
plt.colorbar(label=r"$|\psi|^2$")
plt.xlabel("x")
plt.ylabel("t")
plt.title(r"PINN prediction for $|\psi(x,t)|^2$")
plt.show()

plt.figure(figsize=(8, 5))
plt.plot(loss_history, label="Total loss")
plt.plot(pde_history, label="PDE loss")
plt.plot(ic_history, label="IC loss")
plt.plot(bc_history, label="BC loss")
plt.yscale("log")
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.title("Training losses")
plt.legend()
plt.grid(True)
plt.show()

plt.figure(figsize=(8, 5))
plt.plot(t_eval, norm_t)
plt.xlabel("t")
plt.ylabel(r"$\int |\psi|^2 dx$")
plt.title("Normalization over time")
plt.grid(True)
plt.show()

plt.figure(figsize=(8, 5))
plt.plot(t_eval, x_mean_t)
plt.xlabel("t")
plt.ylabel(r"$\langle x \rangle (t)$")
plt.title("Center of mass over time")
plt.grid(True)
plt.show()

Epoch     0 | Total = 1.467e+00 | PDE = 3.627e-02 | IC = 1.267e-01 | BC = 1.635e-01
Epoch   500 | Total = 3.723e-02 | PDE = 2.539e-02 | IC = 1.082e-03 | BC = 1.026e-03
Epoch  1000 | Total = 2.070e-02 | PDE = 1.716e-02 | IC = 2.499e-04 | BC = 1.035e-03
Epoch  1500 | Total = 1.453e-02 | PDE = 1.108e-02 | IC = 2.036e-04 | BC = 1.419e-03
Epoch  2000 | Total = 1.258e-02 | PDE = 9.826e-03 | IC = 1.210e-04 | BC = 1.549e-03
Epoch  2500 | Total = 1.169e-02 | PDE = 8.458e-03 | IC = 1.230e-04 | BC = 1.999e-03
Epoch  3000 | Total = 1.120e-02 | PDE = 8.055e-03 | IC = 9.693e-05 | BC = 2.175e-03
